# 05 偏りの補正とフィードバックループ

> 🔰 **やさしい導入**（この回のひとこと）
>
> - **ひとことで言うと**: 評価の偏りの補正と、「薦めると、さらに偏りが増える」循環を調べる回です。
> - **たとえ話**: ベストセラー棚に並べるから、さらに売れる。売れるから、ずっと棚に残る。
> - **先に読むと分かりやすい**: [00_basics.ipynb](00_basics.ipynb) の「2-6・2-8（フィードバックループ）」。用語が分からなくなったら、同じNotebookの「用語集」へ。
> - **難しければ**: 各コードの後ろにある「**出力の見方**」の文章だけ読めば、結論は追えます。

> 🔒 すべて**合成データ**です。⏱ 所要時間: 説明 20分 ＋ 実行 約2分半

## ゴール
- **逆確率重み付け（IPS／IPW）** で、評価と学習の偏りを補正できるか試す。**効く場合と、効かない場合**を知る。
- 推薦を**繰り返す**と、薦めたものがよく評価され、**偏りが増幅するか**をシミュレーションで確かめる。
- **人気を抑える並べ替え**（多様性）の代償と、**評価が少ないユーザー**に不利になっていないか（公平性）を見る。

## 前のNotebookとのつながり
01 で、観測データでの評価は**人気に偏る**ことを見ました。03 の ALS は評価の点数を使って個人化できましたが、観測の偏りそのものは直していません。ここでは偏りを**補正**する方法を試し、**その限界**も確かめます。

## キーワード
- **傾向（propensity）**: その商品が観測される確率。ここでは「商品の評価件数 ÷ ユーザー数」で推定する。
- **逆確率重み付け（IPS／IPW）**: 観測される確率の**逆数**を重みにする。「見えにくかった商品」の1件を重く扱う。
- **MNAR**（Missing Not At Random）: 観測されるかどうかが、**好み自体**に依存する。商品ごとの確率では補正しきれない。
- **フィードバックループ**: 薦めたものが、よく評価され、次の学習データになり、また薦められる、という循環。
- **多様性・公平性**: 薦める商品の種類の幅（カバレッジ）、ユーザーの層による成績の差。

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import run_phase5

plt.rcParams["font.family"] = ["Hiragino Sans", "AppleGothic", "DejaVu Sans"]
pd.set_option("display.width", 200)

e1_table, e1_corr = run_phase5.e1()
e1_table

**出力の見方**（評価の補正）:
- 3つの列は、同じ推薦リストを、3通りの方法で測った **recall@10** です。**観測**（テストで高く評価されたものを当てた割合）、**IPS補正**（観測を 1/傾向 で重み付けしたもの）、**本当**（本当の好みで、好きなものを当てた割合）。
- **人気順**は、観測 0.227 でとても良く見えますが、本当は **0.031**（ランダム 0.021 とほぼ同じ）。IPS補正をすると 0.141 まで下がり、**水増しは減りました**。
- 一方、**近傍法（観測 0.242→IPS 0.158、本当 0.056）** と **BPR（一様）（0.231→0.158、本当 0.054）** は、補正しても**高いまま**です。本当に良い **ALS（本当 0.122）** は、補正後も 0.124 と低く見えます。
- 理由: 傾向を「**商品ごと**」にしか推定していないので、**ユーザーの好みによる選ばれやすさ**（好きなものほど評価される＝MNAR）が、補正に入っていません。

In [ ]:
e1_corr

**出力の見方**（手法の並びの一致）: 7つの手法を、観測／IPS補正／本当、それぞれで並べて、**本当の順位との相関**（スピアマン。1 に近いほど同じ並び）を、世界10個で平均しました。
- 偏りが弱い（pop_power 0.5）: 観測 **0.27** に対し、IPS補正 **0.45**。**補正すると少し近づきます**。
- 標準の偏り（1.0）と強い偏り（1.5）: 観測 −0.01／−0.05、IPS補正 0.03／−0.04。**補正しても、本当の順位との相関はほぼゼロ**です。
- 結論: **商品ごとの IPS は、偏りが弱いときには助けになるが、好みに依存した偏り（MNAR）には、効かない**。「補正したから大丈夫」とは言えません。**本物のランダム露出の実験（A/Bテスト）** が、最後の拠り所になります。

In [ ]:
e2 = run_phase5.e2()
e2

**出力の見方**（補正した学習）: ALS の学習で、各評価に **(1/傾向)^γ** の重みを付けました（γ=0 が補正なし）。
- **順位（本当のprecision）**: γ=0 で 0.798、γ=0.25 で 0.800、γ=0.5 で 0.793（標準誤差 0.010〜0.011 なので、**この3つは区別できません**）。γ=1.0 では **0.720 に下がります**（重みが偏りすぎて、数の少ない商品の少数の評価に引きずられる）。
- **全部の組のRMSE**（本当の好みに対する評価点の誤差）は、0.740 → 0.724（γ=0.25）→ 0.720（γ=0.5）と**良くなります**。補正の本来の目的（偏りのない予測）には、**効いている**。
- ただし、上位10件の**順位の良さにはつながらない**。**目的（全体の予測／上位の順位）によって、補正の価値が違う**。
- カバレッジは 0.23 → 0.29（γ=0.5）と、少し広がります。

In [ ]:
e3 = run_phase5.e3()
fig, ax = plt.subplots(1, 4, figsize=(18, 3.8))
for pol, st in [("人気順", "o-"), ("ALS", "s-"), ("ALS＋探索20%", "^-")]:
    g = e3[e3["方針"] == pol]
    for a, col in zip(ax, ["データの評価件数のジニ", "人気上位20pct商品の評価の割合", "薦めた商品のカバレッジ", "固定候補での本当のprecision"]):
        a.plot(g["回"], g[col], st, label=pol); a.set_title(col, fontsize=10); a.set_xlabel("回"); a.grid(alpha=0.3)
ax[0].legend()
plt.tight_layout(); plt.show()
e3[e3["回"].isin([0, 4, 8])]

**出力の見方**（推薦を8回繰り返す）: 毎回、全員に10件を薦め、好きなものほど評価される（本当の好みに基づく確率）。ほかに、自然な閲覧も各人3件ずつある。
- **人気順**: データの評価件数の**偏り（ジニ）は 0.563 → 0.727 に上昇**、人気上位20%の商品が全評価に占める割合は **0.595 → 0.790**。薦めたものがよく評価され、人気の商品がさらに人気になる、**典型的な増幅**です。薦めた商品の種類（カバレッジ）は 0.07 → 0.24 と、少しずつしか増えません。
- **ALS**: 偏りは回 2 で **0.674 までいったん上がり、その後 0.565 まで下がります**（人気上位20%の割合も 0.724 → 0.561）。**増幅し続ける、とは言えません**。カバレッジは 0.24 → 0.68 に広がります。個人の好みで薦めるので、いろいろな商品に評価が集まるためです。
- **ALS＋探索20%**（リストの2件をランダムに入れ替え）: カバレッジは最初から **0.985**。ただし、リストの質は下がる（回 0 の本当のprecision 0.792 → 0.690）。回を重ねても、偏りのピークはALSより低い（0.639）。
- 左の3つ以外に、**右端のグラフ（固定候補での本当のprecision）** は、モデル自体の良さの目安です。**回0の時点で知らなかった商品**だけを候補にして測っています。ALS は 0.784 → 0.930、人気順は 0.310 → 0.373 と、**どちらも上がります**（データが増えるため）。
- ⚠️ 表の「薦めたものの本当のprecision」は、回を重ねると**下がります**（ALS 0.792 → 0.626）。これは偏りの害ではなく、**好きな商品を消費し尽くして、未知の好きな商品が減る**ためです。**方針どうしを、同じ回で比べる**ことが大事で、回をまたいだ比較は注意が必要です。
- まとめ: この設定では、**増幅が顕著なのは、人気順だけ**。ただし、**この結果は、シミュレーションの設定（評価する確率・自然な閲覧の量）に依存**します。現実の増幅の強さを、これで決めてはいけません。

In [ ]:
e4 = run_phase5.e4()
e4

**出力の見方**（人気を抑える並べ替え）: ALS の予測から `λ × log(1+件数)` を引いて並べ替えました（λが大きいほど、人気の商品を強く下げる）。
- λ ≤ 0.1 までは、**本当のprecision はほとんど変わりません**（0.791 → 0.789）。カバレッジは 0.23 → 0.25、新奇性は 7.99 → 8.56 と、**少しだけ多様**になります。**代償が小さい範囲**です。
- λ=0.15 から低下（0.769）、**λ=0.2 で 0.583**、λ=0.4 では **0.196** と急落します。**ある強さを超えると、質が壊れる**。強さは、検証で決めます。
- 多様性（カバレッジ）は、λ=0.2 でも 0.27 にとどまります。**予測の点数の差が大きいので、少しの罰則では、リストが入れ替わらない**ためです。
- **公平性**: 評価が少ないユーザーの precision（0.742）は、多いユーザー（0.822）より **0.08 低い**。λを大きくしても、この差は縮みません（λ=0.1 で 0.731 対 0.830、むしろ拡大）。**人気を抑えるだけでは、不公平は直らない**。評価が少ないユーザーへの対応（04 の新ユーザー対応）が別に要る。

## 試してみよう（演習）
1. **偏りの強さを変える**: `run_phase5.e1(pop_powers=(0.0, 0.25))` を実行してください。分かるのは、偏りがほぼ無いとき、観測の評価と本当の順位がどこまで一致するかです。
2. **IPSの下限**: `item_propensity(..., floor=0.01)` を `0.05` にして `e1` と `e2` を比べてください。分かるのは、重みの爆発を抑えるクリップの効果です。
3. **評価する確率**: `_consume` の `-2.0 * (tru − 3.5)` を `-4.0` にして（好みが評価に強く効く）`e3` を実行してください。分かるのは、増幅が強まるかどうかです。
4. **探索の割合**: `e3` の `tk[x, -2:]` を `-5:`（半分）にして、探索の量とカバレッジ・質のトレードオフを見てください。

## まとめ
- **IPS（商品ごとの傾向）** は、偏りが弱いと助けになる（順位相関 0.27 → 0.45）が、**標準以上の偏りでは効かない**（ほぼ0）。**好みに依存する偏り（MNAR）は、補正できない**。
- **IPW を学習に使う**と、全部の組の予測誤差は良くなる（0.740 → 0.720）が、**上位10件の順位には効かない**。γ=1.0 は有害（0.720）。
- **フィードバックループ**: **人気順では偏りが増幅**（ジニ 0.563 → 0.727）。ALS では、いったん上がって下がる。**ただし、シミュレーションの設定に依存**。
- **人気を抑える並べ替え**は、**弱ければほぼ無料**（λ≤0.1）、**強いと質が壊れる**。**公平性（評価が少ないユーザーへの不利）は、これでは直らない**。

次の Phase6 では、**設定を検証用の世界で固定し、新しい世界で1回だけ最終評価**します（最終レポート）。